# Bank Marketing - Modeling

**DATA 4950 Week 9 Demo**

## What We Will Cover


1. Load preprocessed data
2. Baseline model (Simple Train/Test)
3. Advanced models (Simple Train/Test)
4. Cross-validation
5. Hyperparameter tuning (GridSearchCV)
6. Final model comparision
7. Save the best model

In [ ]:
## import libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## settings
plt.style.use('seaborn-v0_8')
pd.set_option('display.max_columns', None)

In [ ]:
# load data from feature engineering notebook
X_train = pd.read_csv('../data/modeling/X_train.csv')
X_test = pd.read_csv('../data/modeling/X_test.csv')
y_train = pd.read_csv('../data/modeling/y_train.csv').squeeze()
y_test = pd.read_csv('../data/modeling/y_test.csv').squeeze()

## save WITH SMOTE
X_train_smote = pd.read_csv('../data/modeling/X_train_smote.csv')
y_train_smote = pd.read_csv('../data/modeling/y_train_smote.csv').squeeze()

print(f'Training (original): {X_train.shape}')
print(f'Training (SMOTE):    {X_train_smote.shape}')
print(f'Test:                {X_test.shape}')
print(f'\nClass distribution (train): {y_train.value_counts().to_dict()}')
print(f'Class distribution (test):  {y_test.value_counts().to_dict()}')

## PART 1: Basic Models

Start simple. Build complexity gradually.

Train and Test ROC-AUC are printed side by side for every model — a gap > 0.05 suggests overfitting

### model 1: logistic regression

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000, random_state=4950)
lr.fit(X_train, y_train)

In [ ]:
y_pred = lr.predict(X_test)
y_pred

In [ ]:
y_pred[:20]

In [ ]:
y_prob = lr.predict_proba(X_test)[:, 1]   # probability of class 1 (yes)
y_prob[:20]

In [ ]:
## train metrics — to check for overfitting
y_train_pred = lr.predict(X_train)
y_train_prob = lr.predict_proba(X_train)[:, 1]

In [ ]:
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve, ConfusionMatrixDisplay)

In [ ]:
auc_train = roc_auc_score(y_train, y_train_prob)
auc_test = roc_auc_score(y_test, y_prob)
print(f'ROC-AUC  Train: {auc_train:.4f}  |  Test: {auc_test:.4f}  |  Gap: {auc_train-auc_test:.4f}')

In [ ]:
print(classification_report(y_test, y_pred, target_names=['No', 'Yes']))

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=['No','Yes'], colorbar=False)
plt.title('Logistic Regression — Confusion Matrix')
plt.tight_layout()
plt.show()

In [ ]:
results = []

report = classification_report(y_test, y_pred, target_names=['No','Yes'], output_dict=True)
results.append({
    'Model'          : 'Logistic Regression',
    'Accuracy'       : report['accuracy'],
    'Precision (Yes)': report['Yes']['precision'],
    'Recall (Yes)'   : report['Yes']['recall'],
    'F1 (Yes)'       : report['Yes']['f1-score'],
    'ROC-AUC'        : auc_test,
    'ROC-AUC Train'  : auc_train,
})

results

In [ ]:
pd.DataFrame(results).round(4)

In [ ]:
importances = pd.Series(np.abs(lr.coef_[0]), index=X_train.columns)
top15 = importances.sort_values(ascending=False).head(15)

plt.figure(figsize=(10, 6))
top15.plot(kind='barh', color='steelblue', edgecolor='black')
plt.xlabel('Absolute Coefficient Value')
plt.title('Top 15 Features — Logistic Regression')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

**Observations:**

- Train AUC vs Test AUC gap: *(fill in — overfitting?)* 
- Recall (Yes): *(fill in — what % of real depositors did we catch?)*
- Precision (Yes): *(fill in — what % of predicted calls are productive?)*

### model 2: SVM 

Use **scaled** data. `SVC(probability=True)` enables `.predict_proba()` for ROC-AUC.
SVM is evaluated with default parameters only — no GridSearchCV (too slow on large datasets).

In [ ]:
from sklearn.svm import SVC

svm = SVC(probability=True, random_state=4950)
svm.fit(X_train, y_train)

In [ ]:
y_pred = svm.predict(X_test)
y_prob = svm.predict_proba(X_test)[:, 1]

y_train_pred = svm.predict(X_train)
y_train_prob = svm.predict_proba(X_train)[:, 1]

auc_train = roc_auc_score(y_train, y_train_prob)
auc_test = roc_auc_score(y_test, y_prob)
print(f'ROC-AUC  Train: {auc_train:.4f}  |  Test: {auc_test:.4f}  |  Gap: {auc_train-auc_test:.4f}')

In [ ]:
print(classification_report(y_test, y_pred, target_names=['No', 'Yes']))

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, y_pred,
                                        display_labels=['No', 'Yes'],
                                        colorbar=False)
plt.title('SVM — Confusion Matrix')
plt.tight_layout()
plt.show()

In [ ]:
report = classification_report(y_test, y_pred, target_names=['No','Yes'], output_dict=True)
results.append({
    'Model'          : 'SVM',
    'Accuracy'       : report['accuracy'],
    'Precision (Yes)': report['Yes']['precision'],
    'Recall (Yes)'   : report['Yes']['recall'],
    'F1 (Yes)'       : report['Yes']['f1-score'],
    'ROC-AUC'        : auc_test,
    'ROC-AUC Train'  : auc_train,
})

results

In [ ]:
pd.DataFrame(results).round(4)

How does SVM compare to Logistic Regression given both use scaled data? 

### model 3: decision tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier( random_state=4950)
dt.fit(X_train, y_train)

### model 4: random forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=4950, n_jobs=-1)  ##An ensemble of 100 independent decision trees.
rf.fit(X_train, y_train)

How does ensemble performance compare to the single tree?

### model 5: gradient boosting 

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(random_state=4950)
gb.fit(X_train, y_train)

Is Gradient Boosting better or worse than Random Forest?

### Quick comparision

**Observations:**

- Best default model:  
- Models with overfitting (Train/Test gap > 0.05): 
- Overall: is accuracy a reliable metric here? Why or why not?  

## Does SMOTE Help?

 we trained Section 2 models on original data
 
 now retrain the same models on SMOTE data and compare
 
test set stays the same — only training data changes

In [ ]:
## retrain all five models on SMOTE data

lr_s = LogisticRegression(max_iter=1000, random_state=4950)
dt_s = DecisionTreeClassifier(random_state=4950)
svm_s = SVC(probability=True, random_state=4950)
rf_s = RandomForestClassifier(n_estimators=100, random_state=4950, n_jobs=-1)
gb_s = GradientBoostingClassifier(random_state=4950)

lr_s.fit(X_train_smote, y_train_smote)
dt_s.fit(X_train_smote, y_train_smote)
svm_s.fit(X_train_smote, y_train_smote)
rf_s.fit(X_train_smote, y_train_smote)
gb_s.fit(X_train_smote, y_train_smote)

print('Retrained on SMOTE data.')

In [ ]:
## collect AUC for both versions

result_smote = {}
for name, orig, smote in [('Logistic Regression', lr, lr_s),
                           ('Decision Tree',       dt, dt_s),
                           ('SVM',       svm, svm_s),
                           ('Random Forest',       rf, rf_s),
                           ('Gradient Boosting',       gb, gb_s)]:

    auc_orig  = roc_auc_score(y_test, orig.predict_proba(X_test)[:, 1])
    auc_smote = roc_auc_score(y_test, smote.predict_proba(X_test)[:, 1])
    result_smote[name] = {'Original': round(auc_orig, 4),
                     'SMOTE':    round(auc_smote, 4),
                     'Gain':     round(auc_smote - auc_orig, 4)}

smote_comparison = pd.DataFrame(result_smote).T.reset_index()
smote_comparison.columns = ['Model', 'Original AUC', 'SMOTE AUC', 'Gain']

print('SMOTE vs Original — ROC-AUC on Test Set:')
print(smote_comparison)

In [ ]:
## look beyond AUC — check recall on the minority class (Yes = deposit)
## a model that ignores the minority class will have poor recall for Yes

print('=== Without SMOTE ===')
print(classification_report(y_test, lr.predict(X_test), target_names=['No', 'Yes']))

print('=== With SMOTE ===')
print(classification_report(y_test, lr_s.predict(X_test), target_names=['No', 'Yes']))

print('Key insight: compare recall for Yes (row 2) between the two reports')

In [ ]:
## visualize: ROC curves original vs SMOTE for each model

fig, axes = plt.subplots(1, 5, figsize=(25, 5))

for ax, (name, orig, smote) in zip(axes, [('Logistic Regression', lr, lr_s),
                                           ('Decision Tree',       dt, dt_s),
                                           ('SVM',       svm, svm_s),
                                           ('Random Forest',       rf, rf_s),
                                           ('Gradient Boosting',       gb, gb_s) ]):
    for label, model, ls in [('Original', orig, '--'), ('SMOTE', smote, '-')]:
        prob = model.predict_proba(X_test)[:, 1]
        fpr, tpr, _ = roc_curve(y_test, prob)
        auc = roc_auc_score(y_test, prob)
        ax.plot(fpr, tpr, ls=ls, label=f'{label}  (AUC={auc:.3f})')

    ax.plot([0, 1], [0, 1], 'k:', alpha=0.4)
    ax.set_title(name)
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.legend(fontsize=8)

plt.suptitle('SMOTE vs Original — ROC Curves', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
## why doesn't SMOTE help Random Forest?
## RF has a built-in parameter that handles imbalance more effectively

rf_weighted = RandomForestClassifier(n_estimators=100,
                                     class_weight='balanced',   # <-- handles imbalance
                                     random_state=4950, n_jobs=-1)
rf_weighted.fit(X_train, y_train)   # no SMOTE needed

auc_original = roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1])
auc_smote    = roc_auc_score(y_test, rf_s.predict_proba(X_test)[:, 1])
auc_weighted = roc_auc_score(y_test, rf_weighted.predict_proba(X_test)[:, 1])

print(f'RF original:              {auc_original:.4f}')
print(f'RF with SMOTE:            {auc_smote:.4f}')
print(f'RF with class_weight:     {auc_weighted:.4f}')

### Decision — which training set will you use from this point forward?

Note: neither rebalancing approach helped here. The ensemble already handles the 8:1 imbalance without adjustment.class_weight may help on other datasets — always compare empirically.

## Cross-Validation

### Introduce StratifiedKFold

The Problem: Can We Trust One Split?

In [ ]:
## our test AUC came from ONE specific train/test split
## what if Random Forest got lucky and Gradient Boosting got unlucky?
## the difference below looks small — is it real or just noise?

rf_auc = results_df.loc[results_df['Model'] == 'Random Forest', 'ROC-AUC'].iloc[0] 
print(f'Random Forest AUC:      {rf_auc:.4f}')

gb_auc = results_df.loc[results_df['Model'] == 'Gradient Boosting', 'ROC-AUC'].iloc[0] 
print(f'Gradient Boosting AUC:  {gb_auc:.4f}')

print(f'Difference:             {abs(rf_auc- gb_auc):.4f}  <- is this real or noise?')
print()
print('Solution: cross-validation gives us mean ± std across 5 splits')
print('         so we can see if the gap is consistent or accidental')

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

In [ ]:
## stratified = each fold preserves the class ratio (important for imbalanced data)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=4950)

In [ ]:
cv_results = []

## TODO: Choose at least three models to do CV

### TODO: Choose at least three models to do CV

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(cv_df['Model'], cv_df['CV Mean'], xerr=cv_df['CV Std'],
        color='steelblue', edgecolor='black', capsize=4, alpha=0.85)
ax.set_xlabel('CV ROC-AUC (mean ± std)')
ax.set_title('5-Fold Cross-Validation — All Models')
ax.invert_yaxis()
for i, v in enumerate(cv_df['CV Mean']):
    ax.text(v+0.002, i, f'{v:.4f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9,4))
ax.barh(results_df['Model'], results_df['ROC-AUC'], color='steelblue', edgecolor='black')
ax.set_xlabel('ROC-AUC (Test)')
ax.set_title('Default Models — ROC-AUC Comparison')
ax.invert_yaxis()
for i, v in enumerate(results_df['ROC-AUC']):
    ax.text(v+0.002, i, f'{v:.4f}', va='center', fontsize=9)
plt.tight_layout(); plt.show()

### Discussion

**Observations:**

- Do the CV means agree with the single-split test AUCs from Section 2?  
- Which model(s) show CV Std > 0.03? What does that mean? 
- Which models are most consistent across folds?  
- Does CV change your view of which models to tune?  

## Hyperparameter Tuning with GridSearchCV

In [ ]:
from sklearn.model_selection import GridSearchCV

### tuning: logistic regression

- `C`: inverse regularisation strength — smaller C = stronger regularisation = simpler model
- `penalty='l1'`: can zero out weak features (sparse model)
- `penalty='l2'`: shrinks all coefficients proportionally (default)

In [ ]:

## C = inverse regularization strength; l1 can zero out weak features

lr_param_grid = {
    'C':       [0.01, 0.1, 1, 10],
    'penalty': ['l1', 'l2'],
    'solver':  ['liblinear']
}

lr_grid = GridSearchCV(
    LogisticRegression(max_iter=1000, random_state=4950),
    param_grid=lr_param_grid,
    cv=cv, scoring='roc_auc', n_jobs=-1, verbose=1
)
lr_grid.fit(X_train, y_train)

print(f'\nBest params: {lr_grid.best_params_}')
print(f'Best CV AUC: {lr_grid.best_score_:.4f}')

In [ ]:
## inspect lr cv_results_

lr_cv = (pd.DataFrame(lr_grid.cv_results_)
           [['param_C', 'param_penalty', 'mean_test_score', 'std_test_score', 'rank_test_score']]
           .sort_values('rank_test_score')
           .round(4)
           .reset_index(drop=True))

print('All LR parameter combinations:')
print(lr_cv)

In [ ]:
## evaluate tuned logistic regression

y_pred = lr_grid.best_estimator_.predict(X_test)
y_prob = lr_grid.best_estimator_.predict_proba(X_test)[:, 1]
auc_train  = roc_auc_score(y_train, lr_grid.best_estimator_.predict_proba(X_train)[:,1])
auc_test   = roc_auc_score(y_test, y_prob)

print(f'ROC-AUC  Train: {auc_train:.4f}  |  Test: {auc_test:.4f}  |  Gap: {auc_train-auc_test:.4f}')
print(classification_report(y_test, y_pred, target_names=['No','Yes']))

In [ ]:
results

In [ ]:
report = classification_report(y_test, y_pred, target_names=['No','Yes'], output_dict=True)
results.append({
    'Model'          : 'Logistic Regression (Tuned)',
    'Accuracy'       : report['accuracy'], 
    'Precision (Yes)':  report['Yes']['precision'],  
    'Recall (Yes)'   :  report['Yes']['recall'],  
    'F1 (Yes)'       :  report['Yes']['f1-score'],  
    'ROC-AUC'        :  auc_test,  
    'ROC-AUC Train'  :  auc_train, 
})

In [ ]:
pd.DataFrame(results).round(4)

In [ ]:
lr_prob_tuned = y_prob

**Observations:**

- Best params found:  
- Improvement over default LR: 
- Train vs. Test gap:  

### TODO: Complete three models to do GridSearch CV

### tuning: decision tree

In [ ]:
dt_param_grid = {
    'max_depth':        [3, 5, 10, None],
    'min_samples_split':[2, 5, 10],
    'criterion':        ['gini', 'entropy']
}



### tuning: SVM

In [ ]:
## C controls the margin width — larger C = fits training data more tightly
## kernel='rbf' handles non-linear boundaries; 'linear' is faster and simpler
## gamma controls how far the influence of a single training example reaches
# svm_param_grid = {
#     'C':      [0.1, 1, 10],
#     'kernel': ['rbf', 'linear'],
#     'gamma':  ['scale', 'auto']
# }

# svm_grid = GridSearchCV(
#     SVC(probability=True, random_state=4950),
#     param_grid=svm_param_grid,
#     cv=cv, scoring='roc_auc', n_jobs=-1, verbose=1
# )
# svm_grid.fit(X_train, y_train)

# print(f'Best params: {svm_grid.best_params_}')
# print(f'Best CV AUC: {svm_grid.best_score_:.4f}')

In [ ]:
# ## inspect top 5 SVM combinations

# svm_cv = (pd.DataFrame(svm_grid.cv_results_)
#             [['param_C', 'param_kernel', 'param_gamma',
#               'mean_test_score', 'std_test_score', 'rank_test_score']]
#             .sort_values('rank_test_score')
#             .head(5)
#             .round(4)
#             .reset_index(drop=True))

# print('Top 5 SVM parameter combinations:')
# print(svm_cv)

### tuning: random forest

In [ ]:

## max_depth controls overfitting; max_features controls tree diversity

rf_param_grid = {
    'n_estimators':      [100, 200],
    'max_depth':         [None, 10, 20],
    'max_features':      ['sqrt', 'log2'],
    'min_samples_split': [2, 5]
}



### tuning: gradient boosting

In [ ]:
## tuning: gradient boosting
## lower learning_rate needs more trees; subsample < 1.0 adds randomness

gb_param_grid = {
    'n_estimators':  [100, 200],
    'max_depth':     [3, 5],
    'learning_rate': [0.05, 0.1],
    'subsample':     [0.8, 1.0]
}



## Model Selection

In [ ]:
final_df = (pd.DataFrame(results)
              .sort_values('ROC-AUC', ascending=False)
              .reset_index(drop=True))

print(final_df[['Model','Precision (Yes)','Recall (Yes)',
                'F1 (Yes)','ROC-AUC','ROC-AUC Train']].to_string(index=False))

In [ ]:
## Default vs Tuned — comparison from the master results DataFrame

name_map = {
    'Logistic Regression (Tuned)': 'Logistic Regression',
    'Decision Tree (Tuned)'      : 'Decision Tree',
    'Random Forest (Tuned)'      : 'Random Forest',
    'Gradient Boosting (Tuned)'  : 'Gradient Boosting',
}

print(f'{"Model":<35} {"Default AUC":>12} {"Tuned AUC":>10} {"Gain":>8}')
print('-' * 70)

for tuned_name, base_name in name_map.items():
    base_auc  = final_df.loc[final_df['Model'] == base_name,  'ROC-AUC'].values[0]
    tuned_auc = final_df.loc[final_df['Model'] == tuned_name, 'ROC-AUC'].values[0]
    print(f'{tuned_name:<35} {base_auc:>12.4f} {tuned_auc:>10.4f} {tuned_auc - base_auc:>+8.4f}')

In [ ]:
## ROC curves — default vs tuned
## default models: lr, dt, svm, rf, gb  (already fitted in Part 1)
## tuned models: use the saved prob arrays

plt.figure(figsize=(9, 6))

for name, prob in [
    ('LR (Default)',  lr.predict_proba(X_test)[:, 1]),
    ('DT (Default)',  dt.predict_proba(X_test)[:, 1]),
    ('RF (Default)',  rf.predict_proba(X_test)[:, 1]),
    ('GB (Default)',  gb.predict_proba(X_test)[:, 1]),
    ('LR (Tuned)',    lr_prob_tuned),
    ('DT (Tuned)',    dt_prob_tuned),
    ('RF (Tuned)',    rf_prob_tuned),
    ('GB (Tuned)',    gb_prob_tuned),
]:
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc = roc_auc_score(y_test, prob)
    ls  = '-' if '(Tuned)' in name else '--'
    plt.plot(fpr, tpr, ls=ls, label=f'{name}  (AUC={auc:.3f})')

plt.plot([0, 1], [0, 1], 'k:', alpha=0.4, label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves — Default (dashed) vs Tuned (solid)')
plt.legend(fontsize=8, loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
## CV scores for all tuned models — confirms test AUC is not a lucky split
## high std = unstable; prefer low std alongside high mean

print(f'{"Model":<35} {"Mean AUC":>9} {"Std":>7}  Fold Scores')
print('-' * 75)

for name, model in [
    ('Logistic Regression (Tuned)', lr_grid.best_estimator_),
    ('Decision Tree (Tuned)',       dt_grid.best_estimator_),
    ('Random Forest (Tuned)',       rf_grid.best_estimator_),
    ('Gradient Boosting (Tuned)',   gb_grid.best_estimator_),
]:
    scores = cross_val_score(model, X_train, y_train,
                             cv=cv, scoring='roc_auc', n_jobs=-1)
    print(f'{name:<35} {scores.mean():>9.4f} {scores.std():>7.4f}  {np.round(scores, 4)}')

In [ ]:
## feature importance — best model
## update best_model if a different model wins

best_model      = rf_grid.best_estimator_
best_model_name = 'Random Forest (Tuned)'

importances = pd.Series(best_model.feature_importances_, index=X_train.columns)
top15 = importances.sort_values(ascending=False).head(15)

plt.figure(figsize=(10, 6))
top15.plot(kind='barh', color='steelblue', edgecolor='black')
plt.xlabel('Feature Importance')
plt.title(f'Top 15 Features — {best_model_name}')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

### Discussion

**Winning model:**  

Model Results Interpretation:

- Test ROC-AUC:  
- CV mean ± std:  
- Train vs. Test gap: 
- Recall (Yes):  
- Precision (Yes):  
- Improvement over best default model:  
- Top 3 features:  

## Save Best Model

In [ ]:
## save best model
## add to Cell 01 imports
import pickle, os
os.makedirs('../models', exist_ok=True)

with open('../models/best_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)

print(f'Saved:  ../models/best_model.pkl')
print(f'Model:  {type(best_model).__name__}')
print(f'Params: {rf_grid.best_params_}')